# US-001：环境搭建与初识 MNE

**目标：** 安装 MNE-Python，加载示例数据，跑通第一个 EEG 可视化。

**核心对象：** `Raw`、`Info`、`Annotations`

## 1. 环境安装

推荐用 conda 创建独立环境：

In [1]:
# 在终端中执行（不在 notebook 里跑）：
# conda create -n mne python=3.10 -y
# conda activate mne
# pip install mne
# pip install jupyter matplotlib

验证安装：

In [2]:
import mne
mne.sys_info()

Platform             macOS-26.4.1-arm64-arm-64bit-Mach-O
Python               3.14.4 (main, Apr 14 2026, 14:46:33) [Clang 22.1.3 ]
Executable           /Users/usst_ziyi/Programs/QwenPaw/projects/EEG/.venv/bin/python
CPU                  Apple M2 (8 cores)
Memory               8.0 GiB

Core
├☑ mne               1.12.1 (unable to check for latest version on GitHub, unknown error: HTTP Error 403: rate limit exceeded)
├☑ numpy             2.4.4 (unknown linalg bindings)
├☑ scipy             1.17.1
└☑ matplotlib        3.10.9 (backend=module://matplotlib_inline.backend_inline)

Numerical (optional)
├☑ sklearn           1.8.0
├☑ pandas            3.0.2
└☐ unavailable       numba, nibabel, nilearn, dipy, openmeeg, cupy, h5io, h5py

Visualization (optional)
├☑ qtpy              2.4.3 (PyQt6=6.11.0)
├☑ ipympl            0.10.0
├☑ pyqtgraph         0.14.0
├☑ mne-qt-browser    0.7.4
├☑ ipywidgets        8.1.8
└☐ unavailable       pyvista, pyvistaqt, vtk, trame_client, trame_server, trame_vtk, tra

## 2. 加载 MNE 自带示例数据

MNE 提供了一个 sample 数据集，包含听觉/视觉刺激的 EEG + MEG 数据。
`mne.datasets.sample.data_path()` 首次调用会自动下载（约 2GB）。

In [3]:
import mne
import os
# 目录配置
data_dir = os.path.abspath("./../datasets")
os.makedirs(data_dir, exist_ok=True)
# 每次运行强制更新 config
mne.set_config("MNE_DATA", data_dir)
mne.set_config("MNE_DATASETS_SAMPLE_PATH", data_dir)
print("=" * 80)
print("config file =", mne.get_config_path())
print("=" * 80)
# 打印config 内容,友好格式
config = mne.get_config()
for key, value in config.items():
    print(f"* {key:25s}: {value}")


# 获取 sample 数据集路径（首次运行会下载）
sample_data_dir = mne.datasets.sample.data_path()
# 加载原始 EEG 数据（只读 EEG 通道，排除 MEG 和 EOG）
raw_fname = sample_data_dir / "MEG" / "sample" / "sample_audvis_raw.fif"
raw: mne.io.RawArray = mne.io.read_raw_fif(raw_fname, preload=False, verbose=False)
print(raw)

config file = /Users/usst_ziyi/.mne/mne-python.json
* MNE_BROWSE_RAW_SIZE      : 14.277777777777779,11.597222222222221
* MNE_DATA                 : /Users/usst_ziyi/Programs/QwenPaw/projects/EEG/mne-eeg-learning/datasets
* MNE_DATASETS_SAMPLE_PATH : /Users/usst_ziyi/Programs/QwenPaw/projects/EEG/mne-eeg-learning/datasets
* MNE_PATH                 : ./datasets
<Raw | sample_audvis_raw.fif, 376 x 166800 (277.7 s), ~3.2 MiB, data not loaded>


## 3. 认识 Raw 对象

`Raw` 是 MNE 最基础的数据容器，存储连续时间序列数据。

In [4]:
# Raw 对象的关键属性
print(f"通道数: {len(raw.ch_names)}")
print(f"采样率: {raw.info['sfreq']} Hz")
print(f"数据时长: {raw.times[-1]:.1f} 秒")
print(f"前 10 个通道名: {raw.ch_names[:10]}")

通道数: 376
采样率: 600.614990234375 Hz
数据时长: 277.7 秒
前 10 个通道名: ['MEG 0113', 'MEG 0112', 'MEG 0111', 'MEG 0122', 'MEG 0123', 'MEG 0121', 'MEG 0132', 'MEG 0133', 'MEG 0131', 'MEG 0143']


## 4. 认识 Info 对象

`Info` 存储通道元信息：通道类型、位置、采样率、滤波器设置等。

In [5]:
info = raw.info
print(f"MNE 版本: {info.get('mne_version', 'unknown')}")
print(f"高通滤波: {info['highpass']} Hz")
print(f"低通滤波: {info['lowpass']} Hz")

# 查看所有键
print(f"\nInfo 的键: {list(info.keys())}")

MNE 版本: unknown
高通滤波: 0.10000000149011612 Hz
低通滤波: 172.17630004882812 Hz

Info 的键: ['file_id', 'events', 'hpi_results', 'hpi_meas', 'subject_info', 'device_info', 'helium_info', 'hpi_subsystem', 'proc_history', 'meas_id', 'experimenter', 'description', 'proj_id', 'proj_name', 'meas_date', 'utc_offset', 'sfreq', 'highpass', 'lowpass', 'line_freq', 'gantry_angle', 'chs', 'dev_head_t', 'ctf_head_t', 'dev_ctf_t', 'dig', 'bads', 'ch_names', 'nchan', 'projs', 'comps', 'acq_pars', 'acq_stim', 'custom_ref_applied', 'xplotter_layout', 'kit_system_id']


In [6]:
# 按类型统计通道
from collections import Counter

# 直接使用 raw 对象的方法
ch_types = raw.get_channel_types()
print(f"通道类型分布: {dict(Counter(ch_types))}")

通道类型分布: {'grad': 204, 'mag': 102, 'stim': 9, 'eeg': 60, 'eog': 1}


## 5. 认识 Annotations

Annotations 标记时间点/时间段上的事件（如眨眼、刺激 onset、坏段）。

In [7]:
annotations = raw.annotations
print(f"标注数量: {len(annotations)}")
print(f"前 5 个标注:")
print(annotations[:5])

标注数量: 0
前 5 个标注:
<Annotations | 0 segments>


In [8]:
# 如果存在刺激通道，比如 "STI 014"
# events = mne.find_events(raw, stim_channel='STI 014')

# 或者让 MNE 自动找 stim 通道
events = mne.find_events(raw)
print(f"找到的事件数量: {len(events)}")

Finding events on: STI 014
320 events found on stim channel STI 014
Event IDs: [ 1  2  3  4  5 32]
找到的事件数量: 320


## 6. 可视化 EEG 波形

`raw.plot()` 打开交互式波形浏览器，可以缩放、标记坏段。

In [ ]:
%matplotlib qt
# 只画前 10 个 EEG 通道，10 秒数据
eeg_chs = [ch for ch in raw.ch_names if 'EEG' in ch][:10]
raw.pick(eeg_chs)
raw.plot(duration=10, n_channels=10, scalings='auto')

Using qt as 2D backend.


Channels marked as bad:
none


: 

## 7. 小结

| 对象 | 作用 |
|------|------|
| `Raw` | 连续时间序列数据，包含所有通道 |
| `Info` | 元数据：通道名、类型、位置、采样率 |
| `Annotations` | 时间点事件标记（刺激、伪迹等） |

下一步 US-002 将深入 `Raw → Epochs → Evoked` 的数据流转。